In [ ]:
import time
import math

In [ ]:
n = 4096*4096
threads_per_block = 256 #256/32 = 8 warps per block
blocks = int(n / threads_per_block)
grid_size = 256 # 256*256 = 65536 threads in one block

### Technique 1: Grid Stride Loop

**A grid size loop launches a fixed grid and lets each thread stride across the full array, keeping memory coalesced**


Why use this technique?
  * Same kernel works for any size array
  * adjacent threads always read adjacent memory addresses

In [ ]:
import numpy as np
from numba import jit,cuda

#cpu normalization first

@jit(nopython=True)
def cpu_normalization(arr, min_val, max_val):

  out = np.empty_like(arr)
  range_val = max_val - min_val

  for i in range(len(arr)):
    out[i] = (arr[i] - min_val) / range_val

  return out

#gpu normalization naive (1 thread per element)

@cuda.jit
def gpu_normalization_naive(arr, out, min_val, range):

  idx = cuda.grid(1)

  if idx < arr.shape[0]:
    out[idx] = (arr[idx] - min_val) / range

# #gpu normalization stride
@cuda.jit
def gpu_normalization_stride(arr, out, min_val, range):

  start = cuda.grid(1)
  stride = cuda.gridsize(1)
  n = arr.shape[0]
  i = start

  while i < n:
    out[i] = (arr[i] - min_val) / range
    i += stride



## Data and GPU Setup

In [ ]:
range_filler = np.random.default_rng(42)
data = {"values": (range_filler.random(n) * 1000).astype(np.float64)}

arr = data["values"].astype(np.float32)
min_val = float(arr.min())
max_val = float(arr.max())
range_val = max_val - min_val

cpu_normalization(arr[:256], min_val, max_val)

d_arr = cuda.to_device(arr) #CPU RAM to GPU VRAM
d_out = cuda.device_array_like(arr) #allocate empty output on GPU

naive_blocks = math.ceil(len(arr)/ threads_per_block)


## Checking Benchmarks